# FHR-SAC on FetchReach

Gymnasium-Robotics `FetchReachDense-v4` (goal-conditioned Dict obs flattened to Box(16), dense reward = −|grip − goal|, 50-step episodes, γ 0.95) — stock-comparable SAC vs FHRSAC with the **calibrated-FHR pipeline**: a probe arm (λ_eff ≡ 0) measures the TD/penalty magnitude ratio and the converged policy's Q-trace Hankel rank; λ and fhr_order for the real arms come from those measurements. No HER here — the standard multi-goal success-rate protocol (Plappert et al. 2018) needs a HER-aware FHR buffer and is the named follow-up.

## 0 · Launch — train whatever the config defines

In [ ]:
import pathlib, sys
SRC_RUNNERS = pathlib.Path.cwd().parent / "src"
if str(SRC_RUNNERS) not in sys.path:
    sys.path.insert(0, str(SRC_RUNNERS))
import run_sb3_seeds as runner
import yaml

CONFIG = "configs/config_sb3_sac.yaml"
# Every arm below — launched and analysed — is exactly what the config's
# experiment.fhr_experiments block currently defines; nothing is hardcoded.
EXPERIMENTS = sorted(int(k) for k in
                     (yaml.safe_load(open(CONFIG))["experiment"]
                      .get("fhr_experiments") or {}))
print("config experiments:", EXPERIMENTS)

LAUNCH = False
FORCE_EXP = False
if LAUNCH:
    manifest = runner.launch_all(config=CONFIG, experiments=EXPERIMENTS,
                                 max_workers=4, force=FORCE_EXP)
    print(sorted(manifest["runs"]))
else:
    print("LAUNCH = False — analysing existing runs only")

In [ ]:
import csv, json
import numpy as np
import matplotlib.pyplot as plt
import torch

REPO = pathlib.Path.cwd().parents[2]
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False,
})
CFG = yaml.safe_load(open(CONFIG))
CMAN = json.load(open("cached/sb3_runs_manifest_sac.json"))
SEEDS = [str(s) for s in (CFG["experiment"].get("seeds")
                          or [CFG["experiment"]["seed"]])]

# ---- arms are DERIVED from the config, never hardcoded -------------------
PALETTE = ["tab:green", "tab:blue", "tab:orange", "tab:purple", "tab:red",
           "tab:cyan", "peru", "orchid", "crimson", "tab:olive"]
KIND_ORDER = {"baseline": 0, "probe": 1, "global": 2, "ar": 3, "per": 4}

def _spec(n, ov):
    lam = float(ov.get("fhr_weight", CFG["agent"]["fhr_weight"]))
    pred = ov.get("c_predictor")
    per = bool(ov.get("prioritized_replay", False))
    if float(ov.get("warmup_grad_steps", 0)) >= 1e8:
        kind, name = "probe", "probe (lambda_eff = 0)"
    elif per:
        kind, name = "per", "FHR + PER"
    elif pred is None:
        kind, name = "global", "standard FHR (global c)"
    else:
        kind, name = "ar", f"{pred} c(s,a)"
    return {"arm": f"exp{n}", "kind": kind, "lam": lam, "pred": pred,
            "label": f"{name} · λ{lam:g}"}

specs = [{"arm": "baseline", "kind": "baseline", "lam": 0.0, "pred": None,
          "label": "baseline (soft TD only)"}]
unrun = []
for n, ov in sorted((int(k), v) for k, v in
                    (CFG["experiment"].get("fhr_experiments") or {}).items()):
    s = _spec(n, ov)
    (specs if CMAN["runs"].get(s["arm"]) else unrun).append(s)
specs.sort(key=lambda s: (KIND_ORDER[s["kind"]], s["lam"], s["pred"] or ""))
if unrun:
    print("defined but not yet run:", [s["arm"] for s in unrun])

ARMS, INFO = {}, {}
for i, s in enumerate(specs):
    label = s["label"]
    if label in ARMS:
        label = f"{label} [{s['arm']}]"
    s["colour"] = "black" if s["kind"] == "baseline" else \
                  PALETTE[(i - 1) % len(PALETTE)]
    ARMS[label] = (s["arm"], s["colour"])
    INFO[label] = s

def labels_of(*kinds):
    return [l for l in ARMS if INFO[l]["kind"] in kinds]

BASE = "baseline (soft TD only)" if "baseline (soft TD only)" in ARMS else None
GLOBALS_ = labels_of("global")
VARIANTS = labels_of("ar")
PER_ARMS = labels_of("per")

def run_dirs(arm):
    out = []
    for s in SEEDS:
        rel = CMAN["runs"].get(arm, {}).get(s)
        if rel and (pathlib.Path(rel) / "rewards.csv").exists():
            out.append((s, pathlib.Path(rel)))
    return out

def curves(arm, fname="eval.csv", x="env_steps", y="mean_reward"):
    cs = []
    for s, d in run_dirs(arm):
        if not (d / fname).exists():
            continue
        rows = list(csv.DictReader(open(d / fname)))
        if rows:
            cs.append((s, np.array([float(r[x]) for r in rows]),
                       np.array([float(r[y]) for r in rows])))
    return cs

def diag(arm, col):
    return curves(arm, fname="train_diagnostics.csv", x="episode", y=col)

print({l: ARMS[l][0] for l in ARMS})

## 1 · Calibration — λ from error magnitudes, r from converged rank

In [ ]:
# Error-signal calibration (the probe arm trains bit-identically to the
# baseline while logging penalty magnitudes): lambda from the TD/penalty
# ratio, fhr_order from the converged policy's measured Q-trace Hankel rank.
import calibrate_fhr
cal = calibrate_fhr.calibrate(probe_arm="exp9", config=CONFIG,
                              ratios=(0.5, 2.0))
for i, seed in enumerate(cal["seeds"]):
    print(f"seed {seed}: td median {cal['td_median'][i]:.4g}, "
          f"penalty median {cal['penalty_median'][i]:.4g}, "
          f"Q rank {cal['q_rank'][i]}, pi ranks {cal['pi_rank'][i]}")
print("td/penalty ratio:", f"{cal['td_over_penalty']:.3g}")
print("suggested lambda:", {k: round(v, 3) for k, v in cal["lambda"].items()})
print("suggested fhr_order:", cal["fhr_order"])

## 2 · Learning curves

In [ ]:
GROUP_DEFS = [("Baseline vs calibrated FHR", ("baseline", "probe", "global")),
              ("State-conditioned c(s,a)", ("ar",)),
              ("Prioritized replay", ("per",))]
GROUPS = [(t, labels_of(*kinds)) for t, kinds in GROUP_DEFS]
GROUPS = [(t, ls) for t, ls in GROUPS if ls]

def seed_band(ax, label, colr=None, lw=1.9, alpha=0.95, band_alpha=0.16,
              legend=True, zorder=2, **kw):
    cs = curves(ARMS[label][0], **kw)
    if not cs:
        return
    n = min(len(y) for _, _, y in cs)
    x, Y = cs[0][1][:n], np.stack([y[:n] for _, _, y in cs])
    colr = colr or ARMS[label][1]
    if len(cs) > 1:
        ax.fill_between(x, Y.min(0), Y.max(0), color=colr, alpha=band_alpha,
                        lw=0, zorder=zorder - 1)
    ax.plot(x, Y.mean(0), lw=lw, color=colr, alpha=alpha, zorder=zorder,
            label=f"{label} ({len(cs)} seeds)" if legend else None)

fig, axes = plt.subplots(1, len(GROUPS), figsize=(4.9 * len(GROUPS), 4.4),
                         sharey=True, squeeze=False)
for ax, (title, labels) in zip(axes.ravel(), GROUPS):
    if BASE:
        seed_band(ax, BASE, "gray", lw=2.6, alpha=1.0, band_alpha=0.20,
                  legend=BASE in labels, zorder=1)
    for label in labels:
        if label == BASE:
            continue
        seed_band(ax, label)
    ax.axhline(-2, color="gray", ls=":", lw=1)
    ax.set(title=title, xlabel="environment steps")
    ax.legend(fontsize=7.5, loc="lower right")
axes[0, 0].set_ylabel("greedy evaluation return (50-step episodes)")
fig.suptitle(f"{CFG['environment']['name']} greedy-eval curves — seed-mean, "
             f"band = seed min-max (seeds {', '.join(SEEDS)})",
             fontweight="bold", y=1.02)
plt.tight_layout()

## 3 · Final performance

In [ ]:
# Final greedy-eval performance per arm (last eval.csv row, mean over seeds)
rows = []
for label in ARMS:
    cs = curves(ARMS[label][0])
    if cs:
        finals = [y[-1] for _, _, y in cs]
        rows.append((label, np.mean(finals), np.min(finals), np.max(finals),
                     ARMS[label][1]))
fig, ax = plt.subplots(figsize=(8.5, 0.55 * len(rows) + 1.4))
ypos = np.arange(len(rows))[::-1]
for y, (label, mean, lo, hi, colr) in zip(ypos, rows):
    ax.barh(y, mean, color=colr, alpha=0.85, height=0.62)
    ax.plot([lo, hi], [y, y], color="black", lw=1.4)
    ax.text(mean, y, f"  {mean:.1f}", va="center", fontsize=9)
ax.set_yticks(ypos, [r[0] for r in rows], fontsize=9)
ax.set(title="final greedy-eval return (bar = seed mean, whisker = min-max)",
       xlabel="return")
plt.tight_layout()
for label, mean, lo, hi, _ in rows:
    print(f"{label:42s} {mean:9.1f}   [{lo:.1f}, {hi:.1f}]")

## 4 · FHR + SAC internals

In [ ]:
# FHR/SAC internals per gradient-burst row: the recurrence penalty, the
# learned coefficients, the temperature and the actor loss.
panels = [("penalty_raw", "recurrence penalty (raw)", labels_of("global", "ar", "per")),
          ("sum_c", "sum of c (global arms)", GLOBALS_ + PER_ARMS),
          ("c_spread", "c(s,a) spread (ccond arms)", VARIANTS),
          ("companion_radius", "companion spectral radius", GLOBALS_ + PER_ARMS),
          ("ent_coef", "temperature alpha", list(ARMS)),
          ("actor_loss", "actor loss", list(ARMS))]
panels = [(c, t, ls) for c, t, ls in panels if ls]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.4))
for ax, (col, title, labels) in zip(axes.ravel(), panels):
    for label in labels:
        for s, x, y in diag(ARMS[label][0], col):
            m = np.isfinite(y)
            if m.any():
                ax.plot(x[m], y[m], lw=1.2, color=ARMS[label][1],
                        alpha=0.85 if s == SEEDS[0] else 0.5,
                        ls="-" if s == SEEDS[0] else "--",
                        label=label if s == SEEDS[0] else None)
    ax.set(title=title, xlabel="episode")
    if col in ("penalty_raw",):
        ax.set_yscale("log")
    ax.legend(fontsize=6.5)
for ax in axes.ravel()[len(panels):]:
    ax.axis("off")
plt.tight_layout()

## 5 · Rollout Hankel rank — critic trace and the policy itself

In [ ]:
# Rollout Hankel spectra per arm: the critic trace Q(s_t, pi(s_t)) and the
# policy trajectory's action dims. Fetch episodes are 50 steps; 3 rollouts
# stack to (78, 25) Hankels.
from analysis.low_rank.continuous_rollout import hankel_rollout_continuous
from analysis.low_rank.rank import energy_rank

SHOW = [l for l in ([BASE] + labels_of("probe")[:1] + GLOBALS_ + VARIANTS
                    + PER_ARMS) if l]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
table = []
for label in SHOW:
    arm = ARMS[label][0]
    dirs = run_dirs(arm)
    if not dirs:
        continue
    _, adapter = runner.load_run_model(dirs[0][1], device="cpu")
    env = runner._make_env(CFG)
    mats = hankel_rollout_continuous(adapter, env, n_rollouts=3, base_seed=52)
    env.close()
    h_q, h_acts = mats[0], mats[1:]
    ranks = []
    sv = np.linalg.svd(h_q, compute_uv=False); sv = sv / sv[0]
    rq = energy_rank(sv, 0.999); ranks.append(rq)
    axes[0].semilogy(np.arange(1, min(len(sv), 25) + 1), sv[:25], lw=1.8,
                     color=ARMS[label][1], label=f"{label} (rank {rq})")
    pi_ranks = []
    for j, h_a in enumerate(h_acts):
        sva = np.linalg.svd(h_a, compute_uv=False); sva = sva / sva[0]
        ra = energy_rank(sva, 0.999)
        pi_ranks.append(ra)
        if j == 0:
            axes[1].semilogy(np.arange(1, min(len(sva), 25) + 1), sva[:25],
                             lw=1.8, color=ARMS[label][1],
                             label=f"{label} (rank {ra})")
    table.append((label, rq, pi_ranks))
axes[0].set(title="Hankel(Q(s_t, pi(s_t))) — critic trace", xlabel="index",
            ylabel="sigma_i / sigma_1")
axes[1].set(title="Hankel(pi(s_t)[0]) — first action dim", xlabel="index")
for ax in axes:
    ax.legend(fontsize=7)
plt.tight_layout()
print(f"{'arm':46s} rank(Q)  ranks(pi dims)")
for label, rq, pr in table:
    print(f"{label:46s} {rq:7d}  {pr}")

## 6 · GB10 cost

In [ ]:
# GB10 wall-clock per run, from the launcher logs (cached/logs/sb3_sac_*.log
# mtimes bracket each run; the parent prints per-run minutes into its own
# stdout, so here we use the run dirs' filesystem timestamps as a proxy:
# checkpoint mtime - config mtime ~ training duration).
import os
rows = []
for label in ARMS:
    for s, d in run_dirs(ARMS[label][0]):
        t0 = os.path.getmtime(d / "config.yaml")
        t1 = os.path.getmtime(d / "checkpoints" / "final.pt")
        rows.append((label, s, (t1 - t0) / 60))
for label, s, mins in rows:
    print(f"{label:46s} seed {s}: {mins:6.1f} min")
base = [m for l, _, m in rows if l == BASE]
fhr = [m for l, _, m in rows if l != BASE and "probe" not in l]
if base and fhr:
    print(f"\nbaseline mean {np.mean(base):.1f} min; "
          f"FHR-arm mean {np.mean(fhr):.1f} min "
          f"(overhead x{np.mean(fhr)/np.mean(base):.2f})")

## 7 · Findings

*(fill in after the sweep completes)*